In [ ]:
import pandas as pd
from transformers import pipeline
from tqdm import tqdm
import glob

df = pd.read_csv("sampled_game_Reviews.csv")
print(f"Total data: {len(df)} baris")

In [ ]:
# Ganti NaN jadi string kosong, lalu hitung panjangnya
df['char_count'] = df['Review'].fillna("").apply(len)

# Hitung rata-rata
average_chars = df['char_count'].mean()

print(f"Rata-rata karakter per Review: {average_chars:.2f}")


In [ ]:
# 2. Pastikan kolom 'Reviews' ada dan tidak berisi NaN
df = df.dropna(subset=["Review"])

# 3. Ambil 10 sampel acak
sample_Reviews = df["Review"].sample(n=10, random_state=42).reset_index(drop=True)

# 4. Tampilkan hasil
print(sample_Reviews)

In [ ]:
import ssl
import nltk

try:
    _create_unverified_https_context = ssl._create_unverified_context
except AttributeError:
    pass
else:
    ssl._create_default_https_context = _create_unverified_https_context

# Sekarang bisa download
nltk.download('punkt')
nltk.download('stopwords')


In [ ]:
# Install hanya sekali
!pip install nltk spacy
!python -m spacy download en_core_web_sm

In [ ]:
import pandas as pd
import re
import nltk
import spacy

# Load data
df = pd.read_csv("sampled_game_Reviews.csv")
df['Review'] = df['Review'].astype(str)

# Tampilkan jumlah data
print("Jumlah data:", len(df))
df[['Review']].head()


In [ ]:
print("Jumlah data sebelum Case Folding:", df['Review'].notnull().sum())

df['case_folding'] = df['Review'].str.lower()

print("Jumlah data sesudah Case Folding:", df['case_folding'].notnull().sum())
print("Contoh:\nSebelum:", df['Review'][0], "\nSesudah:", df['case_folding'][0])


In [ ]:
print("Jumlah data sebelum Cleansing:", df['case_folding'].notnull().sum())

df['cleansing'] = df['case_folding'].apply(lambda x: re.sub(r'[^a-z\s]', '', x))

print("Jumlah data sesudah Cleansing:", df['cleansing'].notnull().sum())
print("Contoh:\nSebelum:", df['case_folding'][0], "\nSesudah:", df['cleansing'][0])


In [ ]:
normalization_dict = {
    "u": "you", "r": "are", "ur": "your", "pls": "please", "plz": "please",
    "thx": "thanks", "ty": "thank you", "btw": "by the way", "lol": "laughing out loud",
    "lmao": "laughing my ass off", "omg": "oh my god", "wtf": "what the fuck",
    "idk": "i do not know", "imo": "in my opinion", "imho": "in my humble opinion",
    "afaik": "as far as i know", "brb": "be right back", "bbl": "be back later",
    "gg": "good game", "ez": "easy", "nerf": "weaken", "buff": "strengthen",
    "noob": "newbie", "grind": "repetitive task", "op": "overpowered",
    "laggy": "slow connection", "gitgud": "get good", "f2p": "free to play",
    "p2w": "pay to win", "dlc": "downloadable content", "npc": "non player character",
    "fps": "frames per second", "afk": "away from keyboard", "xp": "experience points",
    "lvl": "level", "bossfight": "boss fight", "rng": "random number generator",
    "camping": "staying in one spot", "gank": "ambush attack", "re": "regarding",
    "af": "as fuck", "xd": "bad", "f": "fuck", "ive": "i have", "rp": "roleplay",
    "fckin": "fucking", "tl": "too long", "dr": "didnt read", "lcg": "living card game", 
    "stori": "story", "differ": "different"
}

def normalize(text):
    return ' '.join([normalization_dict.get(word, word) for word in text.split()])

print("Jumlah data sebelum Normalization:", df['cleansing'].notnull().sum())

df['normalization'] = df['cleansing'].apply(normalize)

print("Jumlah data sesudah Normalization:", df['normalization'].notnull().sum())
print("Contoh:\nSebelum:", df['cleansing'][0], "\nSesudah:", df['normalization'][0])


In [ ]:
def tokenize(text):
    return text.split()

print("Jumlah data sebelum Tokenizing:", df['normalization'].notnull().sum())

df['tokenized'] = df['normalization'].apply(tokenize)

print("Jumlah data sesudah Tokenizing:", df['tokenized'].notnull().sum())
print("Contoh:\nSebelum:", df['normalization'][0], "\nSesudah:", df['tokenized'][0])


In [ ]:
pip install spacy

In [ ]:
pip install https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.7.1/en_core_web_sm-3.7.1-py3-none-any.whl

In [ ]:
pip install spacy

In [ ]:
!python -m spacy download en_core_web_sm

In [ ]:
nlp = spacy.load("en_core_web_sm")

def lemmatize(tokens):
    doc = nlp(' '.join(tokens))
    return [token.lemma_ for token in doc]

print("Jumlah data sebelum Lemmatization:", df['tokenized'].notnull().sum())

df['lemmatized'] = df['tokenized'].apply(lemmatize)

print("Jumlah data sesudah Lemmatization:", df['lemmatized'].notnull().sum())
print("Contoh:\nSebelum:", df['tokenized'][0], "\nSesudah:", df['lemmatized'][0])


In [ ]:
from nltk.corpus import stopwords
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

def remove_stopwords(tokens):
    return [word for word in tokens if word not in stop_words]

print("Jumlah data sebelum Stopword Removal:", df['lemmatized'].notnull().sum())

df['final_tokens'] = df['lemmatized'].apply(remove_stopwords)

print("Jumlah data sesudah Stopword Removal:", df['final_tokens'].notnull().sum())
print("Contoh:\nSebelum:", df['lemmatized'][0], "\nSesudah:", df['final_tokens'][0])


In [ ]:
import pandas as pd

# Asumsi df sudah dimuat dari file CSV sebelumnya

# Gabungkan hasil akhir ke dalam kolom teks baru (cleaned_Reviews)
df['cleaned_Reviews'] = df['final_tokens'].apply(lambda x: ' '.join(x))

# Pilih dan urutkan kolom yang akan disimpan
output_columns = [
    'Game',               # nama game (pastikan kolom ini ada di file asli)
    'Review',             # Review asli
    'case_folding',
    'cleansing',
    'normalization',
    'tokenized',
    'lemmatized',
    'final_tokens',       # hasil setelah stopword removal
    'cleaned_Reviews'     # versi string final
]

# Periksa apakah kolom 'game' ada, jika tidak, beri peringatan
if 'game' not in df.columns:
    print("?? Kolom 'game' tidak ditemukan! Pastikan ada di file CSV asli.")
    if 'game' in output_columns:
        output_columns.remove('game')

# Simpan ke CSV
df[output_columns].to_csv("preprocessed_Reviews.csv", index=False)

print("? File berhasil disimpan sebagai 'preprocessed_Reviews.csv'")


In [ ]:
df = pd.read_csv("preprocessed_Reviews.csv")
df.head(10)